# 🛒 Retail Orders Analysis
**Topics Covered:**
1. Orders by Hour — Peak Time Analysis
2. Delay Rate by Hour
3. Impact of Items Count on Fulfillment Time
4. Substitutions Impact
5. Store Performance Comparison
6. Correlation Analysis

---
## Step 1 — Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# ── Style ──────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
STORE_COLORS = {'Houston': '#4C72B0', 'Dallas': '#DD8452', 'Austin': '#55A868'}

# ── Load ───────────────────────────────────────────────────────────────────
df = pd.read_csv('retail_orders.csv', parse_dates=['Order_Time', 'Delivery_Time'])

# ── Feature engineering ────────────────────────────────────────────────────
df['Order_Hour']  = df['Order_Time'].dt.hour
df['Is_Delayed_Bool'] = df['Is_Delayed'].map({'Yes': 1, 'No': 0})

print(f"Dataset shape : {df.shape}")
print(f"Date range    : {df['Order_Time'].min().date()}  →  {df['Order_Time'].max().date()}")
df.head()

In [ ]:
# Quick data-quality check
print("Missing values:\n", df.isnull().sum())
print("\nDtypes:\n", df.dtypes)
df.describe()

---
## Step 2 — Orders by Hour (Peak Time Analysis)

In [ ]:
orders_by_hour = df.groupby('Order_Hour').size().rename('Order_Count')

peak_hour = orders_by_hour.idxmax()
print(f"Peak ordering hour : {peak_hour}:00  ({orders_by_hour.max()} orders)")
print("\nOrders per hour:")
print(orders_by_hour.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

bars = ax.bar(
    orders_by_hour.index,
    orders_by_hour.values,
    color=[
        '#e74c3c' if h == peak_hour else '#4C72B0'
        for h in orders_by_hour.index
    ],
    width=0.7,
    edgecolor='white'
)

ax.bar_label(bars, fmt='%d', padding=3, fontsize=10)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Orders')
ax.set_title('Orders by Hour of Day  (red = peak hour)', fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.set_xlim(orders_by_hour.index.min() - 0.5, orders_by_hour.index.max() + 0.5)

plt.tight_layout()
plt.show()

**Insight:** The bar marked in red is the hour with the highest order volume. Staffing and inventory preparation should be prioritised around this window to prevent bottlenecks.

---
## Step 3 — Delay Rate by Hour

In [ ]:
delay_by_hour = (
    df.groupby('Order_Hour')['Is_Delayed_Bool']
    .agg(['mean', 'sum', 'count'])
    .rename(columns={'mean': 'Delay_Rate', 'sum': 'Delayed_Orders', 'count': 'Total_Orders'})
)
delay_by_hour['Delay_Rate_Pct'] = (delay_by_hour['Delay_Rate'] * 100).round(1)

worst_hour = delay_by_hour['Delay_Rate'].idxmax()
print(f"Hour with highest delay rate : {worst_hour}:00  "
      f"({delay_by_hour.loc[worst_hour, 'Delay_Rate_Pct']}%)")
print("\nDelay summary by hour:")
print(delay_by_hour)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))

# Bar — total orders
ax1.bar(
    delay_by_hour.index,
    delay_by_hour['Total_Orders'],
    color='#AED6F1', label='Total Orders', width=0.6, zorder=2
)
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Total Orders', color='#2980B9')
ax1.tick_params(axis='y', labelcolor='#2980B9')

# Line — delay rate
ax2 = ax1.twinx()
ax2.plot(
    delay_by_hour.index,
    delay_by_hour['Delay_Rate_Pct'],
    color='#e74c3c', marker='o', linewidth=2.5, label='Delay Rate %'
)
ax2.set_ylabel('Delay Rate (%)', color='#e74c3c')
ax2.tick_params(axis='y', labelcolor='#e74c3c')
ax2.set_ylim(0, 110)

# Annotations
for h, row in delay_by_hour.iterrows():
    ax2.annotate(f"{row['Delay_Rate_Pct']}%",
                 xy=(h, row['Delay_Rate_Pct']),
                 xytext=(0, 10), textcoords='offset points',
                 ha='center', fontsize=9, color='#c0392b')

ax1.set_title('Order Volume vs Delay Rate by Hour', fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.show()

**Insight:** High order volume hours do not always correspond to high delay rates — the dual-axis chart separates volume pressure from operational inefficiency so you can pinpoint which hours genuinely need intervention.

---
## Step 4 — Impact of Items Count on Fulfillment Time

In [ ]:
# Bin items into size buckets
df['Items_Bucket'] = pd.cut(
    df['Items_Count'],
    bins=[0, 7, 12, 17, 100],
    labels=['Small (1–7)', 'Medium (8–12)', 'Large (13–17)', 'XL (18+)']
)

bucket_stats = (
    df.groupby('Items_Bucket', observed=True)['Fulfillment_Time_Min']
    .agg(['mean', 'median', 'std', 'count'])
    .round(2)
)

# Pearson correlation
r, p = stats.pearsonr(df['Items_Count'], df['Fulfillment_Time_Min'])
print(f"Pearson r = {r:.3f}   p-value = {p:.4f}")
print("\nFulfillment time by items bucket:")
print(bucket_stats)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Scatter with regression ────────────────────────────────────────────────
for store, grp in df.groupby('Store_Location'):
    ax1.scatter(grp['Items_Count'], grp['Fulfillment_Time_Min'],
                label=store, color=STORE_COLORS[store], alpha=0.8, edgecolors='white', s=70)

m, b = np.polyfit(df['Items_Count'], df['Fulfillment_Time_Min'], 1)
x_line = np.linspace(df['Items_Count'].min(), df['Items_Count'].max(), 100)
ax1.plot(x_line, m * x_line + b, color='black', linewidth=2, linestyle='--', label='Trend')
ax1.set_xlabel('Items Count')
ax1.set_ylabel('Fulfillment Time (min)')
ax1.set_title(f'Items Count vs Fulfillment Time\n(r = {r:.3f}, p = {p:.4f})', fontweight='bold')
ax1.legend()

# ── Box plot by bucket ─────────────────────────────────────────────────────
bucket_data = [df.loc[df['Items_Bucket'] == b, 'Fulfillment_Time_Min']
               for b in df['Items_Bucket'].cat.categories]
bp = ax2.boxplot(bucket_data, patch_artist=True, notch=False,
                  medianprops={'color': 'black', 'linewidth': 2})
colors = ['#AED6F1', '#85C1E9', '#5DADE2', '#2E86C1']
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)

ax2.set_xticklabels(df['Items_Bucket'].cat.categories, rotation=15)
ax2.set_xlabel('Order Size')
ax2.set_ylabel('Fulfillment Time (min)')
ax2.set_title('Fulfillment Time Distribution by Order Size', fontweight='bold')

plt.tight_layout()
plt.show()

**Insight:** A positive correlation between Items Count and Fulfillment Time is expected. The box plots reveal whether the variance also grows with order size — high variance in large orders signals unpredictable picking times worth investigating.

---
## Step 5 — Substitutions Impact

In [ ]:
# Categorise substitution level
df['Sub_Group'] = pd.cut(
    df['Substitutions'],
    bins=[-1, 0, 2, 100],
    labels=['None (0)', 'Low (1–2)', 'High (3+)']
)

sub_stats = (
    df.groupby('Sub_Group', observed=True)
    .agg(
        Order_Count=('Order_ID', 'count'),
        Avg_Fulfillment=('Fulfillment_Time_Min', 'mean'),
        Delay_Rate=('Is_Delayed_Bool', 'mean')
    )
    .round(2)
)
sub_stats['Delay_Rate_Pct'] = (sub_stats['Delay_Rate'] * 100).round(1)
print(sub_stats)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sub_groups = sub_stats.index.tolist()
bar_colors = ['#55A868', '#F0A500', '#e74c3c']

# ── Order count ────────────────────────────────────────────────────────────
bars0 = axes[0].bar(sub_groups, sub_stats['Order_Count'], color=bar_colors, edgecolor='white')
axes[0].bar_label(bars0, fmt='%d', padding=3)
axes[0].set_title('Order Count by Substitution Level', fontweight='bold')
axes[0].set_ylabel('Orders')
axes[0].set_xlabel('Substitution Group')

# ── Avg fulfillment ────────────────────────────────────────────────────────
bars1 = axes[1].bar(sub_groups, sub_stats['Avg_Fulfillment'], color=bar_colors, edgecolor='white')
axes[1].bar_label(bars1, fmt='%.1f min', padding=3)
axes[1].set_title('Avg Fulfillment Time by Substitution Level', fontweight='bold')
axes[1].set_ylabel('Avg Fulfillment Time (min)')
axes[1].set_xlabel('Substitution Group')

# ── Delay rate ─────────────────────────────────────────────────────────────
bars2 = axes[2].bar(sub_groups, sub_stats['Delay_Rate_Pct'], color=bar_colors, edgecolor='white')
axes[2].bar_label(bars2, fmt='%.1f%%', padding=3)
axes[2].set_title('Delay Rate by Substitution Level', fontweight='bold')
axes[2].set_ylabel('Delay Rate (%)')
axes[2].set_xlabel('Substitution Group')
axes[2].set_ylim(0, 110)

plt.suptitle('Impact of Substitutions on Fulfillment & Delays', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Insight:** High-substitution orders tend to have longer fulfillment times and higher delay rates, because staff must locate alternative products. Reducing substitution frequency (better stock management) or pre-staging likely substitutes can cut both metrics.

---
## Step 6 — Store Performance Comparison

In [ ]:
store_stats = (
    df.groupby('Store_Location')
    .agg(
        Total_Orders=('Order_ID', 'count'),
        Avg_Items=('Items_Count', 'mean'),
        Avg_Subs=('Substitutions', 'mean'),
        Avg_Fulfillment=('Fulfillment_Time_Min', 'mean'),
        Median_Fulfillment=('Fulfillment_Time_Min', 'median'),
        Delay_Rate=('Is_Delayed_Bool', 'mean')
    )
    .round(2)
)
store_stats['Delay_Rate_Pct'] = (store_stats['Delay_Rate'] * 100).round(1)

best_delay = store_stats['Delay_Rate'].idxmin()
fastest    = store_stats['Avg_Fulfillment'].idxmin()
print(f"Lowest delay rate  → {best_delay}")
print(f"Fastest avg time   → {fastest}")
print("\nStore summary:")
print(store_stats)

In [ ]:
stores    = store_stats.index.tolist()
s_colors  = [STORE_COLORS[s] for s in stores]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Avg fulfillment ────────────────────────────────────────────────────────
bars0 = axes[0].bar(stores, store_stats['Avg_Fulfillment'], color=s_colors, edgecolor='white')
axes[0].bar_label(bars0, fmt='%.1f min', padding=3)
axes[0].set_title('Avg Fulfillment Time by Store', fontweight='bold')
axes[0].set_ylabel('Minutes')

# ── Delay rate ─────────────────────────────────────────────────────────────
bars1 = axes[1].bar(stores, store_stats['Delay_Rate_Pct'], color=s_colors, edgecolor='white')
axes[1].bar_label(bars1, fmt='%.1f%%', padding=3)
axes[1].set_title('Delay Rate by Store', fontweight='bold')
axes[1].set_ylabel('Delay Rate (%)')
axes[1].set_ylim(0, 110)

# ── Avg substitutions ─────────────────────────────────────────────────────
bars2 = axes[2].bar(stores, store_stats['Avg_Subs'], color=s_colors, edgecolor='white')
axes[2].bar_label(bars2, fmt='%.2f', padding=3)
axes[2].set_title('Avg Substitutions per Order by Store', fontweight='bold')
axes[2].set_ylabel('Substitutions')

plt.suptitle('Store Performance Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Fulfillment distribution per store (violin)
fig, ax = plt.subplots(figsize=(9, 5))

store_order = store_stats.sort_values('Avg_Fulfillment').index.tolist()
data_per_store = [df.loc[df['Store_Location'] == s, 'Fulfillment_Time_Min'] for s in store_order]

parts = ax.violinplot(data_per_store, showmedians=True)
for pc, store in zip(parts['bodies'], store_order):
    pc.set_facecolor(STORE_COLORS[store])
    pc.set_alpha(0.7)
for part in ['cbars', 'cmins', 'cmaxes', 'cmedians']:
    parts[part].set_color('black')

ax.set_xticks(range(1, len(store_order) + 1))
ax.set_xticklabels(store_order)
ax.set_ylabel('Fulfillment Time (min)')
ax.set_title('Fulfillment Time Distribution per Store', fontweight='bold')

plt.tight_layout()
plt.show()

**Insight:** Compare each store not only on average speed but also on variance (violin width). A wide violin means unpredictable performance — even if the average is acceptable, high variance hurts customer experience.

---
## Step 7 — Correlation Analysis

In [ ]:
# Select numeric + encoded columns
corr_df = df[['Items_Count', 'Substitutions', 'Fulfillment_Time_Min', 'Is_Delayed_Bool', 'Order_Hour']].copy()

corr_matrix = corr_df.corr()
print("Correlation matrix:")
print(corr_matrix.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle → hide duplicates
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    mask=mask,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Heatmap', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Pair plot for a visual overview of all pairwise relationships
pair_df = corr_df.rename(columns={
    'Items_Count': 'Items',
    'Substitutions': 'Subs',
    'Fulfillment_Time_Min': 'Fulfillment (min)',
    'Is_Delayed_Bool': 'Delayed',
    'Order_Hour': 'Hour'
})

g = sns.pairplot(
    pair_df,
    hue='Delayed',
    palette={0: '#55A868', 1: '#e74c3c'},
    diag_kind='kde',
    plot_kws={'alpha': 0.6, 'edgecolor': 'none'},
    corner=True
)
g.figure.suptitle('Pairplot — Delayed (red) vs On-time (green)', y=1.02, fontweight='bold')
plt.show()

In [ ]:
# Print the strongest correlations (excluding self-correlations)
unstacked = (
    corr_matrix.where(np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool))
    .stack()
    .rename('correlation')
    .reset_index()
    .rename(columns={'level_0': 'Variable A', 'level_1': 'Variable B'})
    .sort_values('correlation', key=abs, ascending=False)
)

print("Top correlations (by absolute value):")
print(unstacked.to_string(index=False))

**Insight summary:**
- A correlation close to **+1** between `Items_Count` and `Fulfillment_Time_Min` confirms that larger orders reliably take longer.
- A positive correlation between `Substitutions` and `Is_Delayed_Bool` indicates that item unavailability is a driver of delays.
- `Order_Hour` correlations reveal whether late-day orders are systematically slower — useful for shift scheduling.
- The pair plot separates delayed (red) from on-time (green) orders, letting you visually spot which feature combinations predict delays best.

---
## Step 8 — Executive Summary

In [ ]:
total_orders   = len(df)
overall_delay  = df['Is_Delayed_Bool'].mean() * 100
avg_fulfil     = df['Fulfillment_Time_Min'].mean()

print("=" * 52)
print("        RETAIL ORDERS — EXECUTIVE SUMMARY        ")
print("=" * 52)
print(f"  Total orders analysed      : {total_orders}")
print(f"  Overall delay rate         : {overall_delay:.1f}%")
print(f"  Avg fulfillment time       : {avg_fulfil:.1f} min")
print(f"  Peak ordering hour         : {peak_hour}:00")
print(f"  Store with lowest delays   : {best_delay}")
print(f"  Fastest store (avg time)   : {fastest}")
print("=" * 52)
print("\nKey drivers of delays identified:")
print("  1. Higher items count → longer fulfillment")
print("  2. More substitutions → higher delay probability")
print("  3. Delay rate varies meaningfully by hour of day")
print("  4. Store-level differences suggest operational gaps")